# Imports and Data Prep

In [ ]:
import os
import sys
import gc
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import AgglomerativeClustering
from tqdm import tqdm
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from sklearn.decomposition import TruncatedSVD
from scipy.stats import zscore
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import importlib
from collections import defaultdict
import traceback
from matplotlib.backends.backend_pdf import PdfPages
from typing import Union
from joblib import Parallel, delayed
from multiprocessing import Pool, cpu_count

from scipy.ndimage import percentile_filter, gaussian_filter
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import fftconvolve
import pickle
import tifffile
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


In [ ]:
BASE_DIR = Path("/mnt/storage-raid10/Yun/analysis_output/chemogenetic")
PROJ_ID = "hcrt-trpv1_huc-h2b-g8m_csn_120min"
PROJ_CTRL = "huc-h2b-g8m_csn_120min"
EXPT_FISH_LIST = [
    "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2", 
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish3",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1"
]

N_COMPONENTS = 15
N_CLUSTERS = 18
PHASIC_DPRIME_THRESH = 0.25
RESPONSE_TYPES = ["tonic_pos", "tonic_neg", "phasic_pos", "phasic_neg"]

DIR_ANTS_OUTPUT = str(BASE_DIR)

example_fish = '251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4' #used for testing

In [ ]:
#Parameters 

# IMAGING SPECS 
sec_per_volume = 1
volume_per_sec = 1
n_slices = 40
depth = 250
binning = 1
res_x = 1.52*binning
res_y = 1.52*binning
res_z = depth/n_slices
rotation_k = 2 

# DRUG PERFUSION PARAMETER 
drug_uM   = 10.0
V_ml     = 15.0
Q_ml_min = 4.5


baseline_start = 0 * 60 * volume_per_sec
baseline_end = 45 * 60 * volume_per_sec

# define drug perfusion start & end time frame 
drug_start = 46  * 60 * volume_per_sec
drug_end = 90  * 60 * volume_per_sec

# define E3+DMSO wash out start & end time frame
wash_start = 91 * 60 * volume_per_sec
wash_end = 120 * 60 * volume_per_sec


#  define delta F / F 's baseline percentile
df_f_percentile = 20

# define F_tonic's window size and percentile
f_tonic_window_size = 600  #seconds
f_tonic_percentile = 20

# permutation test parameters
p_thresh_permutation = 0.005
n_resample_permutation = 500

# RUN BH-FDR FILTERING CODE
BH_Q = 0.05  



input_tag    = "C"
K_global     = 600
drift_global = 1       # Order 1 (Linear)
lam_global   = 0.5
lag_global   = 0  

param_folder_name = f"in{input_tag}_K{K_global}_drift{drift_global}_lam{lam_global}_lag{lag_global}"

CLIP_ABS_DZ = 50.0 
INCLUDED_BASELINE = 15.0     # minutes of baseline to use (matches GLM fit_baseline_sec)
NULL_TAG = "iaaft"
RESPONDER_NULL_THRESH = 95  # threshold for null
L_MIN = 20.0   # plateau duration in minutes



In [ ]:
#low mem method to load fish data and responders

def process_fish(fish_id):
    fish_dir = BASE_DIR / PROJ_ID / fish_id
    
    try:
        f_tonic = np.load(fish_dir / "f_tonic.npy", mmap_mode="r")
        f_phasic = np.load(fish_dir / "f_phasic.npy", mmap_mode="r")
        
      
        tonic_pos_idx = np.load(fish_dir / "tonic_pos_glm_iaaft_nullp95_idxs.npy")
        tonic_neg_idx = np.load(fish_dir / "tonic_neg_glm_iaaft_nullp95_idxs.npy")
        dprime = np.load(fish_dir / "phasic_dprime_cells_raw.npy")

      
        all_tonic_idx = np.union1d(tonic_pos_idx, tonic_neg_idx)
        all_phasic_idx = np.where(np.abs(dprime) >= PHASIC_DPRIME_THRESH)[0]

     
        tonic_slice = np.array(f_tonic[all_tonic_idx, :])
        phasic_slice = np.array(f_phasic[all_phasic_idx, :])

        return fish_id, tonic_slice, phasic_slice

    except FileNotFoundError:
        return None

# Main Execution
tonic_data = {}
phasic_data = {}

print("Started processing fish folders")


with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish, EXPT_FISH_LIST)

for result in results:
    if result is not None:
        fish_id, t_res, p_res = result
        tonic_data[fish_id] = t_res
        phasic_data[fish_id] = p_res

print(f"Completed processing for {len(tonic_data)} fish.")


In [ ]:
'''

#For later with control

ctrl_ID_1 = ''
ctrl_ID_2 = ''
ctrl_ID_3 = ''
ctrl_ID_4 = ''
ctrl_ID_5 = ''
ctrl_ID_6 = ''
ctrl_ID_7 = ''

#ctrl_expt_ID_list = [ctrl_ID_1, ctrl_ID_2, ctrl_ID_3, ctrl_ID_4, ctrl_ID_5, ctrl_ID_6, ctrl_ID_7]

expt_ID_list = ctrl_expt_ID_list + expt_fish_list

'''

In [ ]:
from scipy.stats import zscore

z_phasic = {}

for fish_id in EXPT_FISH_LIST:
  
    if fish_id in phasic_data:
    
        raw_phasic_traces = phasic_data[fish_id]

        z_phasic_traces = zscore(raw_phasic_traces, axis=1)

        z_phasic[fish_id] = z_phasic_traces

    else:
        print(f"Fish data matrix not found in memory for: {fish_id}\n")

print("All experimental fish traces have been successfully normalized!")


# Hyperparameter Optimization


## Scree Plot for determining optimal number of factors

In [ ]:
#Scree Plot for determining optimal number of factors
def plot_scree_curve(raw_phasic_traces, target_fish):

    try:
        clean_raw_traces = raw_phasic_traces[np.var(raw_phasic_traces, axis=1) > 0]
        n_dropped = len(raw_phasic_traces) - len(clean_raw_traces)

        if n_dropped > 0:
            print(f"Automatically filtered out {n_dropped} dead/flat cells with zero variance.")

        z_phasic_traces = zscore(clean_raw_traces, axis=1)
        data_matrix = z_phasic_traces.T
        print(f"Evaluating factor spectrum for clean matrix shape: {data_matrix.shape}...")

        n_factors = 30 #max number of factor it will test
        svd = TruncatedSVD(n_components=n_factors, random_state=42)
        svd.fit(data_matrix)

        plt.figure(figsize=(10, 5))
        plt.plot(range(1, n_factors + 1), svd.explained_variance_ratio_, 'o-', linewidth=2, color='#1f77b4')
        plt.axvline(x=20, color='r', linestyle='--', label="Baseline # of factors (K=20)")
        plt.title(f"Scree for {target_fish}", fontsize=14)
        plt.xlabel("Number of Factors", fontsize=12)
        plt.ylabel("Explained Variance Ratio", fontsize=12)
        plt.xticks(range(1, n_factors + 1))
        plt.legend(loc="upper right")
        plt.grid(True, alpha=0.3)
        plt.show()
        
    except Exception as e:
        print(f"Error occurred while plotting scree curve for {target_fish}: {e}")


print("Started processing fish folders")

for fish_id in EXPT_FISH_LIST:
    if fish_id in phasic_data:
        raw_phasic_traces = phasic_data[fish_id]
        plot_scree_curve(raw_phasic_traces, fish_id)
    else:
        print(f"Fish data matrix not found in memory for: {fish_id}\n")

print(f"Completed processing for {len(tonic_data)} fish.")


As shown in the elbow plots for the 9 expt fish, the k (number of factors ) where the EVR was at a solid point and adding more clusters yielded diminishing returns was on average 5. 

## Elbow Plot for determining optimal number of clusters

In [ ]:
#Elbow Plot for determining optimal number of clusters
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.cluster import KMeans

def plot_cluster_elbow_curve(raw_phasic_traces, target_fish):
    try:
        clean_raw_traces = raw_phasic_traces[np.var(raw_phasic_traces, axis=1) > 0]
        n_dropped = len(raw_phasic_traces) - len(clean_raw_traces)
        if n_dropped > 0:
            print(f"Automatically filtered out {n_dropped} dead/flat cells with zero variance.")
            
        z_phasic_traces = zscore(clean_raw_traces, axis=1)
        
        # KMEANS clusters rows, rows are cells, so we are good!
    
        data_matrix = z_phasic_traces
        print(f"Evaluating clusters for matrix shape: {data_matrix.shape}...")
        
        max_clusters = 15  #number of max clusters to test for the elbow method
        wcss = []          
        
        
        cluster_range = range(1, max_clusters + 1)
        for k in cluster_range:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=5)
            kmeans.fit(data_matrix)
            wcss.append(kmeans.inertia_) # inertia_ is the WCSS
            
      
        plt.figure(figsize=(10, 5))
        plt.plot(cluster_range, wcss, 'o-', linewidth=2, color='blue')
        plt.title(f"K-Means Elbow Plot for {target_fish}", fontsize=14)
        plt.xlabel("Number of Clusters (K)", fontsize=12)
        plt.ylabel("Within-Cluster Sum of Squares (Inertia)", fontsize=12)
        plt.xticks(cluster_range)
        plt.grid(True, alpha=0.3)
        plt.show()
        
    except Exception as e:
        print(f"Error occurred while plotting elbow curve for {target_fish}: {e}")

print("Started processing fish folders")
for fish_id in EXPT_FISH_LIST:
    if fish_id in phasic_data:
        raw_phasic_traces = phasic_data[fish_id]
        plot_cluster_elbow_curve(raw_phasic_traces, fish_id)
    else:
        print(f"Fish data matrix not found in for: {fish_id}\n")


## Silhouette scores and plots for verifying number of clusters

In [ ]:
def plot_silhouette_curve(raw_phasic_traces, target_fish):
    try:
  
        clean_raw_traces = raw_phasic_traces[np.var(raw_phasic_traces, axis=1) > 0]
        z_phasic_traces = zscore(clean_raw_traces, axis=1) 
        
        # 2. Factor Analysis 
        n_optimal_factors = 15
        fa = FactorAnalysis(n_components=n_optimal_factors, random_state=42)
        # data_matrix will correctly be shape: (n_cells, n_optimal_factors)
        data_matrix = fa.fit_transform(z_phasic_traces)
        
        print(f"Evaluating cell silhouette profile across factors for matrix: {data_matrix.shape}")
        
        min_clusters = 2
        max_clusters = 30
        silhouette_scores = []
        cluster_range = list(range(min_clusters, max_clusters + 1))
        
        for k in cluster_range:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=5)
            cluster_labels = kmeans.fit_predict(data_matrix)
            
            score = silhouette_score(data_matrix, cluster_labels, sample_size=10000, random_state=42)
            silhouette_scores.append(score)
            
        if silhouette_scores:
            best_idx = int(np.argmax(silhouette_scores))
            best_k = cluster_range[best_idx]
            best_score = silhouette_scores[best_idx]
            print(f'Highest silhouette score is for {best_k} clusters with a score of {best_score:.3f}')
            
            # 4. Plotting
            plt.figure(figsize=(10, 5))
            plt.plot(cluster_range, silhouette_scores, 'o-', linewidth=2, color='#9467bd')
            plt.title(f"Silhouette Profile for {target_fish} Cells (Factor Space)", fontsize=14)
            plt.xlabel("Number of Clusters (K)", fontsize=12)
            plt.ylabel("Average Silhouette Score", fontsize=12)
            plt.xticks(cluster_range)
            plt.grid(True, alpha=0.3)
            plt.show()
            
            return best_k, data_matrix
            
    except Exception as e:
        print(f"Error occurred while plotting silhouette curve for {target_fish}: {e}")
        return None, None

# Loop
print("Started processing fish folders")
for fish_id in EXPT_FISH_LIST:
    if fish_id in phasic_data:
        plot_silhouette_curve(phasic_data[fish_id], fish_id)
    else:
        print(f"Fish data matrix not found for: {fish_id}\n")

In [ ]:
# ...existing code...
def find_silhouette_scores(raw_phasic_traces, target_fish):
    clean_raw_traces = raw_phasic_traces[np.var(raw_phasic_traces, axis=1) > 0]
    z_phasic_traces = zscore(clean_raw_traces, axis=1) 

    n_optimal_factors = 15
    fa = FactorAnalysis(n_components=n_optimal_factors, random_state=42)
    data_matrix = fa.fit_transform(z_phasic_traces)

    print(f"Evaluating cell silhouette profile across factors for matrix: {data_matrix.shape}")

    min_clusters = 2
    max_clusters = 30
    cluster_range = list(range(min_clusters, max_clusters + 1))
    silhouette_scores = []
    for n_clusters in cluster_range:
        clusterer = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
        cluster_labels = clusterer.fit_predict(data_matrix)

        clf = NearestCentroid()
        clf.fit(data_matrix, cluster_labels)
        print("Centroids:")
        print(clf.centroids_)

        # safe sample size
        n_samples = data_matrix.shape[0]
        sample_size = min(50000, n_samples)
        if sample_size < n_samples:
            score = silhouette_score(data_matrix, cluster_labels, sample_size=sample_size, random_state=42)
        else:
            score = silhouette_score(data_matrix, cluster_labels)
        silhouette_scores.append(score)

    # after collecting all scores
    best_idx = int(np.argmax(silhouette_scores))
    best_k = cluster_range[best_idx]
    best_score = silhouette_scores[best_idx]
    print(f'Highest silhouette score is for {best_k} clusters with a score of {best_score:.3f}')

    plt.figure(figsize=(10, 5))
    plt.plot(cluster_range, silhouette_scores, 'o-', linewidth=2, color='#9467bd')
    plt.title(f"Silhouette Profile for {target_fish} Cells (Factor Space)", fontsize=14)
    plt.xlabel("Number of Clusters (K)", fontsize=12)
    plt.ylabel("Average Silhouette Score", fontsize=12)
    plt.xticks(cluster_range)
    plt.grid(True, alpha=0.3)
    plt.show()

    return best_k, data_matrix
# ...existing code...

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_samples, silhouette_score
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D
from sklearn.neighbors import NearestCentroid

def find_silhouette_scores(raw_phasic_traces, target_fish):
    clean_raw_traces = raw_phasic_traces[np.var(raw_phasic_traces, axis=1) > 0]
    z_phasic_traces = zscore(clean_raw_traces, axis=1) 

    n_optimal_factors = 15
    fa = FactorAnalysis(n_components=n_optimal_factors, random_state=42)
    data_matrix = fa.fit_transform(z_phasic_traces)

    print(f"Evaluating cell silhouette profile across factors for matrix: {data_matrix.shape}")

    min_clusters = 2
    max_clusters = 30
    cluster_range = list(range(min_clusters, max_clusters + 1))
    silhouette_scores = []

    # Compute the silhouette scores for each sample

    for n_clusters in cluster_range:
        clusterer = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
        cluster_labels = clusterer.fit_predict(data_matrix)

        # safe sample size
        n_samples = data_matrix.shape[0]
        sample_size = min(50000, n_samples)
        if sample_size < n_samples:
            score = silhouette_score(data_matrix, cluster_labels, sample_size=sample_size, random_state=42)
        else:
            score = silhouette_score(data_matrix, cluster_labels)
        silhouette_scores.append(score)

    # after collecting all scores
    best_idx = int(np.argmax(silhouette_scores))
    best_k = cluster_range[best_idx]
    best_score = silhouette_scores[best_idx]
    print(f'Highest silhouette score is for {best_k} clusters with a score of {best_score:.3f}')

    plt.figure(figsize=(10, 5))
    plt.plot(cluster_range, silhouette_scores, 'o-', linewidth=2, color='#9467bd')
    plt.title(f"Silhouette Profile for {target_fish} Cells (Factor Space)", fontsize=14)
    plt.xlabel("Number of Clusters (K)", fontsize=12)
    plt.ylabel("Average Silhouette Score", fontsize=12)
    plt.xticks(cluster_range)
    plt.grid(True, alpha=0.3)
    plt.show()

    return best_k, data_matrix

print("Started processing fish folders")
for fish_id in EXPT_FISH_LIST:
    if fish_id in phasic_data:
        find_silhouette_scores(phasic_data[fish_id], fish_id)
    else:
        print(f"Fish data matrix not found for: {fish_id}\n")



After calculating the average among the 9 best n_clusters, the best n_clusters was about 20. However, because one fish had a very low score for 20, 18 was chosen, as it achieved fair results across all of the fish.

## Summary:

Optimal number of factors: 5 |
Optimal number of clusters: 7

# Step 1: Pipeline for the FA and Clustering 

## New Version

Skip this unless making changes

In [ ]:
import os
import gc
from pathlib import Path
import numpy as np
import joblib
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, dendrogram
from matplotlib.backends.backend_pdf import PdfPages

print("Step one of the clustering notebook (FA + Agglomerative Hierarchical Clustering)")


for fish_id in EXPT_FISH_LIST:
    fish_path = BASE_DIR / PROJ_ID / fish_id
    out_dir = fish_path / f"FA_agglo_clustering_results"
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\nProcessing Directory: {fish_id}")
    print(out_dir.resolve())

    for category in RESPONSE_TYPES:
        try:
            if "tonic" in category:
                trace_file = fish_path / "f_tonic.npy"
                if not trace_file.exists(): continue
                traces = np.load(trace_file)
                
                idx_file = fish_path / f"{category}_glm_iaaft_nullp95_idxs.npy"
                if not idx_file.exists(): continue
                cell_idxs = np.load(idx_file)
                selected_traces = traces[cell_idxs]
            else:
                trace_file = fish_path / "f_phasic.npy"
                if not trace_file.exists(): continue
                traces = np.load(trace_file)
                
                dprime_file = fish_path / "phasic_dprime_cells_raw.npy"
                if not dprime_file.exists(): continue
                dprime = np.load(dprime_file)
                
                if category == "phasic_pos":
                    cell_idxs = np.where(dprime >= PHASIC_DPRIME_THRESH)[0]
                else:
                    cell_idxs = np.where(dprime <= -PHASIC_DPRIME_THRESH)[0]
                selected_traces = traces[cell_idxs]

            if len(cell_idxs) < N_COMPONENTS:
                print(f" Skipped {category} because cell count is too low.")
                continue

     
            valid_mask = np.std(selected_traces, axis=1) > 0
            clean_traces = selected_traces[valid_mask]
            clean_idxs = cell_idxs[valid_mask]
            
            z_traces = zscore(clean_traces, axis=1)
            print(f"{category}: Clustering {z_traces.shape[0]} cells across timeline.")


            fa = FactorAnalysis(n_components=N_COMPONENTS, random_state=0)
            latent_space = fa.fit_transform(z_traces) 
            

            joblib.dump(fa, out_dir / f"FA_model_{category}.joblib")

            factor_variance = np.var(latent_space, axis=0)
            fig, ax = plt.subplots(figsize=(6, 4))
            ax.plot(np.arange(1, N_COMPONENTS + 1), factor_variance, marker='o', color='purple')
            ax.set_title(f"Factor Variance — {category} — {fish_id}")
            ax.set_xlabel("Factor Index")
            ax.set_ylabel("Variance Value")
            fig.tight_layout()
            plt.savefig(out_dir / f"factors_variance_{category}.png", dpi=150)
            plt.close()

            fig, ax = plt.subplots(figsize=(10, 5))
            for i in range(N_COMPONENTS):
                ax.plot(fa.components_[i], label=f"F{i+1}", alpha=0.7)
            ax.set_title(f"Factor Time Traces — {category} — {fish_id}")
            ax.set_xlabel("Timepoint Volumes")
            ax.set_ylabel("Component Weight Loading")
            ax.legend(loc='upper right', fontsize=6, ncol=5)
            fig.tight_layout()
            plt.savefig(out_dir / f"factor_time_traces_{category}.png", dpi=200)
            plt.close()

            pdf_path = out_dir / f"top10cells_allfactors_{category}.pdf"
            with PdfPages(pdf_path) as pdf:
                for i in range(min(5, N_COMPONENTS)):
                    factor_trace = fa.components_[i]
                    correlations = np.array([
                        np.corrcoef(z_traces[j], factor_trace)[0, 1]
                        if np.all(np.isfinite(z_traces[j])) else -np.inf
                        for j in range(z_traces.shape[0])
                    ])
                    top_cell_picks = np.argsort(correlations)[-10:]
                    
                    fig, ax = plt.subplots(figsize=(6, 4))
                    for j in top_cell_picks:
                        ax.plot(z_traces[j], alpha=0.7)
                    ax.set_title(f"Top 10 Cells — Factor {i+1} — {category}")
                    ax.set_xlabel("Timepoint")
                    ax.set_ylabel("Z-score Signal")
                    fig.tight_layout()
                    pdf.savefig(fig)
                    plt.close(fig)

            agglo = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward')
            cluster_labels = agglo.fit_predict(latent_space) + 1 
            
            np.save(out_dir / f"cluster_labels_{category}.npy", cluster_labels)
            np.save(out_dir / f"selected_cell_idxs_{category}.npy", clean_idxs)


            if latent_space.shape[0] <= 30000:
                plot_data = latent_space.copy()
                if plot_data.shape[0] > 5000:
                    np.random.seed(0)  # For reproducibility
                    sampled_picks = np.random.choice(plot_data.shape[0], 5000, replace=False)
                    plot_data = plot_data[sampled_picks]
                    
                linkage_tree = linkage(plot_data, method='ward')
                fig, ax = plt.subplots(figsize=(10, 5))
                dendrogram(linkage_tree, no_labels=True, ax=ax)
                ax.set_title(f"Hierarchical Clustering Dendrogram — {fish_id} — {category}")
                fig.tight_layout()
                plt.savefig(out_dir / f"dendrogram_{category}.png", dpi=200)
                plt.close()

            print(f" {category}: Partitioned cells cleanly. Plots saved to disk.")
            
        except Exception as e:
            print(f" Error executing pipeline category {category}: {e}")
            
    gc.collect()


print("finished fa and clustering for all fish")

In [ ]:

def process_fish_analysis(fish_id):
    fish_dir = BASE_DIR / PROJ_ID / fish_id
    
    try:
        f_tonic = np.load(fish_dir / "f_tonic.npy", mmap_mode="r")
        f_phasic = np.load(fish_dir / "f_phasic.npy", mmap_mode="r")
        
      
        tonic_pos_idx = np.load(fish_dir / "tonic_pos_glm_iaaft_nullp95_idxs.npy")
        tonic_neg_idx = np.load(fish_dir / "tonic_neg_glm_iaaft_nullp95_idxs.npy")
        dprime = np.load(fish_dir / "phasic_dprime_cells_raw.npy")

      
        all_tonic_idx = np.union1d(tonic_pos_idx, tonic_neg_idx)
        all_phasic_idx = np.where(np.abs(dprime) >= PHASIC_DPRIME_THRESH)[0]

     
        tonic_slice = np.array(f_tonic[all_tonic_idx, :])
        phasic_slice = np.array(f_phasic[all_phasic_idx, :])


        return fish_id, tonic_slice, phasic_slice

    except FileNotFoundError:
        return None




with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish_analysis, EXPT_FISH_LIST)

for result in results:
    if result is not None:
        fish_id, t_res, p_res = result
        tonic_data[fish_id] = t_res
        phasic_data[fish_id] = p_res


In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram

target_fish = '251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4'

cluster_dir = BASE_DIR / PROJ_ID / target_fish / "FA_agglo_clustering_results"
category = "phasic_pos"

try:

    cluster_labels = np.load(cluster_dir / f"cluster_labels_{category}.npy")
    clean_idxs = np.load(cluster_dir / f"selected_cell_idxs_{category}.npy")

    raw_traces = np.load(BASE_DIR / PROJ_ID / target_fish / "f_phasic.npy")[clean_idxs]
    from scipy.stats import zscore
    z_traces = zscore(raw_traces, axis=1)
    
    from sklearn.decomposition import FactorAnalysis
    fa = FactorAnalysis(n_components=20, random_state=0)
    latent_space = fa.fit_transform(z_traces)

    print(f"factor footprint data space shape: {latent_space.shape}")

    # 1 Factor Variance Spectrum
    factor_variance = np.var(latent_space, axis=0)
    plt.figure(figsize=(8, 4))
    plt.plot(np.arange(1, 21), factor_variance, marker='o', linewidth=2, color='purple')
    plt.title(f"Factor Variance Spectrum: {category} ({target_fish})", fontsize=13)
    plt.xlabel("Factor Index", fontsize=11)
    plt.ylabel("Variance Accounted For", fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.show()


    #Factor Time Traces (Global Temporal Dynamics)

    plt.figure(figsize=(14, 5))
    # Let's plot the top 5 dominant trends to keep the graph clean and legible
    for i in range(5):
        plt.plot(fa.components_[i], label=f"Factor trend {i+1}", alpha=0.8, linewidth=1.5)
    plt.title(f"Latent Factor Time Traces — Core Global Firing Trajectories: {category}", fontsize=13)
    plt.xlabel("Recording Timeline Frame Volumes", fontsize=11)
    plt.ylabel("Component Amplitude Weight", fontsize=11)
    plt.axvline(x=2700, color='black', linestyle='--', alpha=0.5)
    plt.axvline(x=5400, color='black', linestyle='--', alpha=0.5)
    plt.legend(loc='upper right', ncol=5, fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.show()


    # Tree Dendrogram Projection (Agglomerative Hierarchical Clustering)


    plot_data = latent_space.copy()
    if plot_data.shape[0] > 3000:
        sampled_picks = np.random.choice(plot_data.shape[0], 3000, replace=False)
        plot_data = plot_data[sampled_picks]
        
    print("Computing Ward linkage tree matrix. Please wait...")
    linkage_tree = linkage(plot_data, method='ward')
    
    plt.figure(figsize=(14, 6))
    dendrogram(linkage_tree, no_labels=True, color_threshold=None)
    plt.title(f"Agglomerative Hierarchical Clustering Tree Dendrogram: {category}", fontsize=13)
    plt.xlabel("Segmented Individual Brain Neurons (Leaves)", fontsize=11)
    plt.ylabel("Ward Linkage Dissimilarity Distance", fontsize=11)
    plt.show()

except Exception as e:
    print(f"Error compiling notebook graphics: {e}")


## Previous Version

## Skip if using run_pipeline.py

In [ ]:
import os
import sys
import gc
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import nibabel as nib
import ants
import numpy as np

from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import AgglomerativeClustering
from tqdm import tqdm
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from sklearn.decomposition import TruncatedSVD
from scipy.stats import zscore
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import importlib
from collections import defaultdict
import traceback
from matplotlib.backends.backend_pdf import PdfPages
from typing import Union
from joblib import Parallel, delayed
from multiprocessing import Pool, cpu_count

from scipy.ndimage import percentile_filter, gaussian_filter
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import fftconvolve
import pickle
import tifffile
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


BASE_DIR = Path("/mnt/storage-raid10/Yun/analysis_output/chemogenetic")
PROJ_ID = "hcrt-trpv1_huc-h2b-g8m_csn_120min"
PROJ_CTRL = "huc-h2b-g8m_csn_120min"
EXPT_FISH_LIST = [
    "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2", 
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish3",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1"
]

N_COMPONENTS = 5
N_CLUSTERS = 7
PHASIC_DPRIME_THRESH = 0.25
RESPONSE_TYPES = ["tonic_pos", "tonic_neg", "phasic_pos", "phasic_neg"]

DIR_ANTS_OUTPUT = str(BASE_DIR)

example_fish = '251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4' #used for testing

#Parameters 

# IMAGING SPECS 
sec_per_volume = 1
volume_per_sec = 1
n_slices = 40
depth = 250
binning = 1
res_x = 1.52*binning
res_y = 1.52*binning
res_z = depth/n_slices
rotation_k = 2 

# DRUG PERFUSION PARAMETER 
drug_uM   = 10.0
V_ml     = 15.0
Q_ml_min = 4.5


baseline_start = 0 * 60 * volume_per_sec
baseline_end = 45 * 60 * volume_per_sec

# define drug perfusion start & end time frame 
drug_start = 46  * 60 * volume_per_sec
drug_end = 90  * 60 * volume_per_sec

# define E3+DMSO wash out start & end time frame
wash_start = 91 * 60 * volume_per_sec
wash_end = 120 * 60 * volume_per_sec


#  define delta F / F 's baseline percentile
df_f_percentile = 20

# define F_tonic's window size and percentile
f_tonic_window_size = 600  #seconds
f_tonic_percentile = 20

# permutation test parameters
p_thresh_permutation = 0.005
n_resample_permutation = 500

# RUN BH-FDR FILTERING CODE
BH_Q = 0.05  



input_tag    = "C"
K_global     = 600
drift_global = 1       # Order 1 (Linear)
lam_global   = 0.5
lag_global   = 0  

param_folder_name = f"in{input_tag}_K{K_global}_drift{drift_global}_lam{lam_global}_lag{lag_global}"

CLIP_ABS_DZ = 50.0 
INCLUDED_BASELINE = 15.0     # minutes of baseline to use (matches GLM fit_baseline_sec)
NULL_TAG = "iaaft"
RESPONDER_NULL_THRESH = 95  # threshold for null
L_MIN = 20.0   # plateau duration in minutes



### Original

In [ ]:
def process_single_experiment(args):
    expt_ID, PROJ_ID, DIR_ANTS_OUTPUT, medoid_types, stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components = args
    print(f"\nExperiment: {expt_ID}")
    
    dir_expt = os.path.join(DIR_ANTS_OUTPUT, PROJ_ID, expt_ID)
    out_dir = os.path.join(dir_expt, f"FA_agglo_clustering_f={n_components}_c={t}_t={t}")
    os.makedirs(out_dir, exist_ok=True)
    
    for label in medoid_types:
        is_tonic = "tonic" in label

        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = os.path.join(dir_expt, data_filename)
        
        if not os.path.exists(data_path):
            print(f"Skipped {label} — missing data: {data_path}")
            continue
            
        traces = np.load(data_path)
        
     
        if is_tonic:
            idx_filename = f"{label}_glm_iaaft_nullp95_idxs.npy"
            idx_path = os.path.join(dir_expt, idx_filename)
            if not os.path.exists(idx_path):
                print(f"Skipped {label} — missing index file: {idx_path}")
                continue
            idxs = np.load(idx_path)
        else:
            dprime_path = os.path.join(dir_expt, "phasic_dprime_cells_raw.npy")
            if not os.path.exists(dprime_path):
                print(f"Skipped {label} — missing dprime file: {dprime_path}")
                continue
            dprime = np.load(dprime_path)
            if "pos" in label:
                idxs = np.where(dprime >= PHASIC_DPRIME_THRESH)[0]
            else: # "neg"
                idxs = np.where(dprime <= -PHASIC_DPRIME_THRESH)[0]
                
        print(f" {label} — {expt_ID}: Loaded {len(idxs)} indices, trace shape = {traces.shape}")
        
        # Ensure array indices fall within valid trace bounds
        idxs = idxs[idxs < traces.shape[0]]
        if len(idxs) == 0:
            print(f"No valid indices for {label} in {expt_ID}")
            continue
            
        trace_subset = traces[idxs, stim_start:stim_end]
        z_traces = zscore(trace_subset, axis=1)
        valid_mask = np.all(np.isfinite(z_traces), axis=1)
        
        if np.sum(valid_mask) < 4:
            print(f"Too few valid cells after z-score in {expt_ID}—{label}, skipping.")
            continue
            
        z_traces = z_traces[valid_mask]
        idxs = idxs[valid_mask]
        print(f" {label} — {expt_ID}: {z_traces.shape[0]} cells after QC/z-score")
        
        MAX_CLUSTER_CELLS = 50000
        if z_traces.shape[0] > MAX_CLUSTER_CELLS:
            selected_idx = np.random.choice(z_traces.shape[0], MAX_CLUSTER_CELLS, replace=False)
            z_traces = z_traces[selected_idx]
            idxs = idxs[selected_idx]
            print(f" Truncated to {z_traces.shape[0]} cells for clustering")
            
        print(f"Proceeding to FA + clustering with {z_traces.shape[0]} cells")
        
        try:
            fa = FactorAnalysis(n_components=n_components, random_state=0)
            latent = fa.fit_transform(z_traces)
            
            fa_model_path = os.path.join(out_dir, f"FA_model_{label}.joblib")
            joblib.dump(fa, fa_model_path)
            print(f"Saved FA model to {fa_model_path}")
            
            
            del z_traces, latent, trace_subset, idxs
            gc.collect()
            
        except Exception as e:
            print(f"Clustering failed for {label} in {expt_ID} — {e}")
            continue
            
    return expt_ID


def hierarchical_clustering_sequential(
    EXPT_FISH_LIST, PROJ_ID, DIR_ANTS_OUTPUT, RESPONSE_TYPES, stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components
):
    for expt_ID in tqdm(EXPT_FISH_LIST, desc="Sequential clustering"):
        try:
            process_single_experiment((
                expt_ID, PROJ_ID, DIR_ANTS_OUTPUT, RESPONSE_TYPES, 
                stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components
            ))
            print(f"Finished clustering: {expt_ID}")
        except Exception as e:
            print(f"Error in {expt_ID}: {e}")
            import traceback; traceback.print_exc()

if __name__ == "__main__":
    hierarchical_clustering_sequential(
        EXPT_FISH_LIST = EXPT_FISH_LIST,
        PROJ_ID = PROJ_ID,
        DIR_ANTS_OUTPUT = DIR_ANTS_OUTPUT,
        RESPONSE_TYPES = RESPONSE_TYPES,
        stim_start = 2700, 
        stim_end = 7200,
        use_qc = False,
        t = N_CLUSTERS,
        save_dendrogram = True,
        max_cells = 5000,
        n_components = N_COMPONENTS
    )
    

### Optimized Speed

In [ ]:

from multiprocessing import Pool, cpu_count

def process_single_experiment(args):
    expt_ID, PROJ_ID, DIR_ANTS_OUTPUT, medoid_types, stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components = args
    
    dir_expt = os.path.join(DIR_ANTS_OUTPUT, PROJ_ID, expt_ID)
    out_dir = os.path.join(dir_expt, f"FA_agglo_clustering_f={n_components}_c={t}_t={t}")
    os.makedirs(out_dir, exist_ok=True)
    
    loaded_traces = {}

    for label in medoid_types:
        is_tonic = "tonic" in label
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = os.path.join(dir_expt, data_filename)
        
        if not os.path.exists(data_path):
            continue

        if data_filename not in loaded_traces:
            loaded_traces[data_filename] = np.load(data_path)
        traces = loaded_traces[data_filename]

        if is_tonic:
            idx_filename = f"{label}_glm_iaaft_nullp95_idxs.npy"
            idx_path = os.path.join(dir_expt, idx_filename)
            if not os.path.exists(idx_path):
                continue
            idxs = np.load(idx_path)
        else:
            dprime_path = os.path.join(dir_expt, "phasic_dprime_cells_raw.npy")
            if not os.path.exists(dprime_path):
                continue
            dprime = np.load(dprime_path)
            if "pos" in label:
                idxs = np.where(dprime >= PHASIC_DPRIME_THRESH)[0]
            else:
                idxs = np.where(dprime <= -PHASIC_DPRIME_THRESH)[0]

        idxs = idxs[idxs < traces.shape[0]]
        if len(idxs) == 0:
            continue
            
        trace_subset = traces[idxs, stim_start:stim_end]
        z_traces = zscore(trace_subset, axis=1)
        valid_mask = np.all(np.isfinite(z_traces), axis=1)
        
        if np.sum(valid_mask) < 4:
            continue
            
        z_traces = z_traces[valid_mask]
        idxs = idxs[valid_mask]

        MAX_CLUSTER_CELLS = 50000
        if z_traces.shape[0] > MAX_CLUSTER_CELLS:
            selected_idx = np.random.choice(z_traces.shape[0], MAX_CLUSTER_CELLS, replace=False)
            z_traces = z_traces[selected_idx]
            idxs = idxs[selected_idx]

        print(f"Proceeding to FA + clustering with {z_traces.shape[0]} cells")
        try:
            fa = FactorAnalysis(n_components=n_components, random_state=0)
            latent = fa.fit_transform(z_traces)
            fa_model_path = os.path.join(out_dir, f"FA_model_{label}.joblib")
            joblib.dump(fa, fa_model_path)
            
            del z_traces, latent, trace_subset, idxs
        except Exception:
            continue
        
    del loaded_traces
    gc.collect()
    return expt_ID

def hierarchical_clustering_parallel(
    EXPT_FISH_LIST, PROJ_ID, DIR_ANTS_OUTPUT, RESPONSE_TYPES, stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components
):
    tasks = [(
        expt_ID, PROJ_ID, DIR_ANTS_OUTPUT, RESPONSE_TYPES, stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components
    ) for expt_ID in EXPT_FISH_LIST]
    
    num_workers = max(1, int(cpu_count() * 0.75)) 
    
    print(f"Starting parallel execution with {num_workers} workers.")
    
    # progress bar
    with Pool(processes=num_workers) as pool:
        results = list(tqdm(pool.imap_unordered(process_single_experiment, tasks), total=len(tasks), desc="Parallel clustering"))
        
    return results

if __name__ == "__main__":
    hierarchical_clustering_parallel(
        EXPT_FISH_LIST = EXPT_FISH_LIST,
        PROJ_ID = PROJ_ID,
        DIR_ANTS_OUTPUT = DIR_ANTS_OUTPUT,
        RESPONSE_TYPES = RESPONSE_TYPES,
        stim_start = 2700,
        stim_end = 7200,
        use_qc = False,
        t = N_CLUSTERS,
        save_dendrogram = True,
        max_cells = 5000,
        n_components = N_COMPONENTS
    )


## load output from Factor Analysis and Hierarchical Clustering

In [ ]:
stim_start = 2700
stim_end = 7200

loaded_cluster_data = {fish: {cat: {} for cat in RESPONSE_TYPES} for fish in EXPT_FISH_LIST}

def process_fish_analysis(fish_id):
    fish_dir = BASE_DIR / PROJ_ID / fish_id
    cluster_dir = fish_dir / f"FA_agglo_clustering_f={N_COMPONENTS}_c={N_CLUSTERS}_t={N_CLUSTERS}"
    
    if not cluster_dir.exists():
        print(f"Skipping {fish_id} — Clustering output directory not found.")
        return None
        
    fish_results = {cat: {} for cat in RESPONSE_TYPES}
    
    for category in RESPONSE_TYPES:
        is_tonic = "tonic" in category
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = fish_dir / data_filename
        
        if not data_path.exists():
            continue
            
        try:
            all_cluster_cells = []
            cluster_cell_mappings = {}
            
            for c_idx in range(1, N_CLUSTERS + 1):
                idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
                if idx_file.exists():
                    c_cells = np.load(idx_file)
                    if len(c_cells) > 0:
                        all_cluster_cells.append(c_cells)
                        cluster_cell_mappings[c_idx] = c_cells
                        
            if not all_cluster_cells:
                continue
                
            total_responders = np.concatenate(all_cluster_cells)
            
    
            unique_responders, inverse_mapping = np.unique(total_responders, return_inverse=True)
        
         
            raw_traces_mmap = np.load(data_path, mmap_mode="r")
            pooled_post_drug_traces = np.array(raw_traces_mmap[unique_responders, stim_start:stim_end])
        
            cell_to_ram_idx = {cell: i for i, cell in enumerate(unique_responders)}
            
            from scipy.stats import zscore
            for c_idx in range(1, N_CLUSTERS + 1):
                if c_idx not in cluster_cell_mappings:
                    continue
                
                cluster_cells = cluster_cell_mappings[c_idx]
                ram_indices = [cell_to_ram_idx[cell] for cell in cluster_cells]
                
                c_traces = pooled_post_drug_traces[ram_indices]
                z_c_traces = zscore(c_traces, axis=1)

                valid_mask = np.all(np.isfinite(z_c_traces), axis=1)
                z_c_traces = z_c_traces[valid_mask]
                
                if len(z_c_traces) > 0:
                    fish_results[category][c_idx] = z_c_traces
                    
        except Exception as e:
            print(f"Error loading {category} for {fish_id}: {e}")
            continue
            
    return fish_id, fish_results
with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish_analysis, EXPT_FISH_LIST)


for result in results:
    if result is not None:
        fish_id, fish_results = result
        loaded_cluster_data[fish_id] = fish_results
        
print("Data loading completed!")


In [ ]:
loaded_cluster_data['251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4']

In [ ]:
import os
from pathlib import Path

# Target fish directory
check_fish_id = "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4"
target_fish_dir = Path(BASE_DIR) / PROJ_ID / check_fish_id

print(f"=== DIAGNOSTIC REPORT FOR {check_fish_id} ===")
print(f"1. Does fish root folder exist? {target_fish_dir.exists()}")

if target_fish_dir.exists():
    print("\n2. Subdirectories inside this fish folder:")
    subdirs = [x.name for x in target_fish_dir.iterdir() if x.is_dir()]
    for s in subdirs:
        print(f"   - {s}")
        
    # Check the exact directory your loader is trying to find
    loader_target_dir = target_fish_dir / f"FA_agglo_clustering_f={N_COMPONENTS}_c={N_CLUSTERS}_t={N_CLUSTERS}"
    print(f"\n3. Exact folder your loader is looking for:\n   {loader_target_dir}")
    print(f"   Does it exist? {loader_target_dir.exists()}")
    
    if loader_target_dir.exists():
        print("\n4. Index files inside that folder:")
        files = list(loader_target_dir.glob("*.npy"))
        if files:
            for f in files[:5]:
                print(f"   - {f.name}")
        else:
            print("   ⚠️ No .npy index files found in this folder!")


In [ ]:
import os
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from scipy.stats import zscore

stim_start_post = 2700
stim_end_post = 7200

loaded_cluster_data = {fish: {cat: {} for cat in RESPONSE_TYPES} for fish in EXPT_FISH_LIST}

def process_fish_analysis(fish_id):
    fish_dir = BASE_DIR / PROJ_ID / fish_id
    cluster_dir = fish_dir / f"FA_agglo_clustering_f={N_COMPONENTS}_c={N_CLUSTERS}_t={N_CLUSTERS}"
    
    if not cluster_dir.exists():
        print(f"Skipping {fish_id} — Clustering output directory not found.")
        return None
        
    fish_results = {cat: {} for cat in RESPONSE_TYPES}
    
    for category in RESPONSE_TYPES:
        is_tonic = "tonic" in category
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = fish_dir / data_filename
        
        if not data_path.exists():
            continue
            
        try:
            all_cluster_cells = []
            cluster_cell_mappings = {}
            
            for c_idx in range(1, N_CLUSTERS + 1):
                idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
                if idx_file.exists():
                    c_cells = np.load(idx_file)
                    if len(c_cells) > 0:
                        all_cluster_cells.append(c_cells)
                        cluster_cell_mappings[c_idx] = c_cells
                        
            if not all_cluster_cells:
                continue
                
            total_responders = np.concatenate(all_cluster_cells)
            unique_responders, inverse_mapping = np.unique(total_responders, return_inverse=True)
    
            raw_traces_mmap = np.load(data_path, mmap_mode="r")
            
            pooled_post_traces = np.array(raw_traces_mmap[unique_responders, stim_start_post:stim_end_post])
            pooled_full_traces = np.array(raw_traces_mmap[unique_responders, 0:7200])
            
            current_idx_pointer = 0
            for c_idx in range(1, N_CLUSTERS + 1):
                if c_idx not in cluster_cell_mappings:
                    continue
                    
                n_cells_in_cluster = len(cluster_cell_mappings[c_idx])
                cluster_inverse_subset = inverse_mapping[current_idx_pointer : current_idx_pointer + n_cells_in_cluster]
                current_idx_pointer += n_cells_in_cluster
                
                c_post = pooled_post_traces[cluster_inverse_subset]
                c_full = pooled_full_traces[cluster_inverse_subset]
                

                z_post = zscore(c_post, axis=1)
                z_full = zscore(c_full, axis=1)
                
              
                valid_mask = np.all(np.isfinite(z_post), axis=1) & np.all(np.isfinite(z_full), axis=1)
                
                if np.sum(valid_mask) > 0:
                    fish_results[category][c_idx] = {
                        'post_drug': z_post[valid_mask],
                        'full_timeline': z_full[valid_mask]
                    }
                    
        except Exception as e:
            print(f"Error loading {category} for {fish_id}: {e}")
            continue
            
    return fish_id, fish_results

print("Loading Dual-Window post-drug and full-timeline matrices from disk...")
with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish_analysis, EXPT_FISH_LIST)

for result in results:
    if result is not None:
        fish_id, fish_results = result
        loaded_cluster_data[fish_id] = fish_results
        
print("Data loading completed successfully!")


In [ ]:
TEMPLATE_BRAIN_PATH = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz"
SUMMARY_PLOTS_DIR = "/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary"
CATEGORY_COLORS = {
    'tonic_pos': 'orange',
    'tonic_neg': 'blue', 
    'phasic_pos': 'red',   
    'phasic_neg': 'purple'
}

LABEL_TITLES = { 
    'tonic_pos': 'Tonic(+)', 
    'tonic_neg': 'Tonic(-)', 
    'phasic_pos': 'Phasic(+)', 
    'phasic_neg': 'Phasic(-)' 
}

print("Initializing Global Reference Brain Canvas for Raw Responders...")
nii_obj = nib.load(TEMPLATE_BRAIN_PATH)
template_img = nii_obj.get_fdata()
bg_canvas = np.max(template_img, axis=2).T
bg_canvas = np.flipud(bg_canvas)

plt.ioff()

print("Generating Pre-Clustering Raw Responder Map Overlays...")

for expt_ID in EXPT_FISH_LIST:
    print(f"\nEvaluating raw responder maps for: {expt_ID}")
    fish_dir = BASE_DIR / PROJ_ID / expt_ID
    vox_path = fish_dir / "medoids_template_vox.npy"
    if not vox_path.exists():
        continue
    medoids_vox = np.load(vox_path)
    valid_registration_mask = (medoids_vox >= 0).all(axis=1)
    
    for category in RESPONSE_TYPES:
        target_subfolder = os.path.join(SUMMARY_PLOTS_DIR, expt_ID, category)
        os.makedirs(target_subfolder, exist_ok=True)
        
        is_tonic = "tonic" in category
        if is_tonic:
            idx_path = fish_dir / f"{category}_glm_iaaft_nullp95_idxs.npy"
            if not idx_path.exists():
                continue
            raw_responder_idxs = np.load(idx_path)
        else:

            dprime_path = fish_dir / "phasic_dprime_cells_raw.npy"
            if not dprime_path.exists():
                continue
            dprime = np.load(dprime_path)
            if "pos" in category:
                raw_responder_idxs = np.where(dprime >= 0.30)[0]
            else:
                raw_responder_idxs = np.where(dprime <= -0.30)[0]
            
        valid_raw_idxs = raw_responder_idxs[valid_registration_mask[raw_responder_idxs]]
        total_cells_found = len(valid_raw_idxs)
        
        if total_cells_found == 0:
            continue
    
        coords = medoids_vox[valid_raw_idxs]

        fig, ax = plt.subplots(figsize=(8, 8))
    
        ax.imshow(bg_canvas, cmap="gray", origin="upper")
    
        ax.scatter(
            coords[:, 0], 
            bg_canvas.shape[0] - coords[:, 1], 
            color=CATEGORY_COLORS[category],
            s=1.5,     
            alpha=0.4,     
            edgecolors='none'
        )
      
        ax.set_title(f"intermediate plot— {LABEL_TITLES[category]}\n{expt_ID} (Total n = {total_cells_found} cells)", fontsize=12)
        ax.set_xlim(0, bg_canvas.shape[1])
        ax.set_ylim(bg_canvas.shape[0], 0)
        ax.axis("off")
        

        output_path = os.path.join(target_subfolder, "raw_responder_distribution.png")
        fig.savefig(output_path, dpi=200, bbox_inches='tight', pad_inches=0.3)
        plt.close(fig)
        
    gc.collect()


plt.ion()


In [ ]:
TEMPLATE_BRAIN_PATH = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz"
SUMMARY_PLOTS_DIR = "/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary"

CATEGORY_COLORS = {
    'tonic_pos': 'orange',
    'tonic_neg': 'blue',    
    'phasic_pos': 'red',   
    'phasic_neg': 'purple'
}

LABEL_TITLES = { 
    'tonic_pos': 'Tonic(+)', 
    'tonic_neg': 'Tonic(-)', 
    'phasic_pos': 'Phasic(+)', 
    'phasic_neg': 'Phasic(-)' 
}

print("Initializing Global Reference Brain Canvas for Raw Responders...")
if not os.path.exists(TEMPLATE_BRAIN_PATH):
    raise FileNotFoundError(f"Missing mandatory anatomical canvas template: {TEMPLATE_BRAIN_PATH}")

nii_obj = nib.load(TEMPLATE_BRAIN_PATH)
template_img = nii_obj.get_fdata()
bg_canvas = np.max(template_img, axis=2).T
bg_canvas = np.flipud(bg_canvas)

plt.ioff()

print("Generating Pre-Clustering Raw Responder Map Overlays...")

for expt_ID in EXPT_FISH_LIST:
    print(f"\nEvaluating raw responder maps for: {expt_ID}")
    fish_dir = BASE_DIR / PROJ_ID / expt_ID

    vox_path = fish_dir / "medoids_template_vox.npy"
    if not vox_path.exists():
        print(f" Warning: {vox_path} missing. Skipping this fish.")
        continue
    medoids_vox = np.load(vox_path)

    valid_registration_mask = (medoids_vox >= 0).all(axis=1)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    plt.suptitle(f"Pre-Clustering Whole Responder Distributions\n{expt_ID} (Tonic: p95 | Phasic: ±0.25 Cutoff)", fontsize=15, y=0.98)
    axes = axes.flatten()
    
    for idx, category in enumerate(RESPONSE_TYPES):
        ax = axes[idx]

        is_tonic = "tonic" in category
        if is_tonic:
            idx_path = fish_dir / f"{category}_glm_iaaft_nullp95_idxs.npy"
            if not idx_path.exists():
                ax.text(0.5, 0.5, f"Missing index file:\n{category}_nullp95", ha='center', va='center', color='gray')
                ax.axis("off")
                continue
            raw_responder_idxs = np.load(idx_path)
        else:
            dprime_path = fish_dir / "phasic_dprime_cells_raw.npy"
            if not dprime_path.exists():
                ax.text(0.5, 0.5, f"Missing dprime file:\nphasic_dprime_cells_raw.npy", ha='center', va='center', color='gray')
                ax.axis("off")
                continue
            dprime = np.load(dprime_path)
            if "pos" in category:
                raw_responder_idxs = np.where(dprime >= 0.25)[0]
            else:
                raw_responder_idxs = np.where(dprime <= -0.25)[0]
       
        valid_raw_idxs = raw_responder_idxs[valid_registration_mask[raw_responder_idxs]]
        total_cells_found = len(valid_raw_idxs)
        
        ax.imshow(bg_canvas, cmap="gray", origin="upper")
        
        if total_cells_found == 0:
            ax.text(0.5, 0.5, f"No Responding Cells\n(n = 0)", ha='center', va='center', color='gray')
            ax.set_title(LABEL_TITLES[category], fontsize=12)
            ax.axis("off")
            continue
            
        coords = medoids_vox[valid_raw_idxs]
        
    
        ax.scatter(
            coords[:, 0], 
            bg_canvas.shape[0] - coords[:, 1], 
            color=CATEGORY_COLORS[category],
            s=1.2,    
            alpha=0.35,    
            edgecolors='none'
        )
    
        ax.set_title(f"{LABEL_TITLES[category]} Responder Pool (n = {total_cells_found} cells)", fontsize=13)
        ax.set_xlim(0, bg_canvas.shape[1])
        ax.set_ylim(bg_canvas.shape[0], 0)
        ax.axis("off")

    fig.subplots_adjust(wspace=0.3, hspace=0.3)
    fish_summary_dir = os.path.join(SUMMARY_PLOTS_DIR, expt_ID)
    os.makedirs(fish_summary_dir, exist_ok=True)
    
    output_path = os.path.join(fish_summary_dir, "0_whole_pool_responder_distribution.png")
    fig.savefig(output_path, dpi=200, bbox_inches='tight', pad_inches=0.4)
    print(f"  Successfully saved unified raw pool grid -> {output_path}")
    
    plt.close(fig)
    gc.collect()

plt.ion()
print("\nAll intermediate whole pool responder distribution maps rendered and linked inside your sidebar workspace!")


In [ ]:
print("Medoids shape:", medoids_vox.shape)
print("Dprime shape:", dprime.shape)


In [ ]:
# UPDATE THESE CORE STRATEGY VARIABLES AT THE TOP OF STEP 1
N_CLUSTERS = 7 # Lowering from 18 or 7 down to 4 aggregates fragments into clean regional circuits!
PHASIC_DPRIME_THRESH = 0.25  # Realignment to Yun's target parameter


In [ ]:

TEMPLATE_BRAIN_PATH = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz"
SUMMARY_PLOTS_DIR = "/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary"
CONSOLIDATED_PALETTE = ['#FF1493', '#00FFFF', '#FFD700', '#00FF00'] 
nii_obj = nib.load(TEMPLATE_BRAIN_PATH)
bg_canvas = np.max(nii_obj.get_fdata(), axis=2).T
bg_canvas = np.flipud(bg_canvas)

plt.ioff()

for expt_ID in EXPT_FISH_LIST:
    fish_dir = BASE_DIR / PROJ_ID / expt_ID
    vox_path = fish_dir / "medoids_template_vox.npy"
    if not vox_path.exists(): continue
    
    medoids_vox = np.load(vox_path)
    valid_mask = (medoids_vox >= 0).all(axis=1)
    

    cluster_dir = fish_dir / f"FA_agglo_clustering_f={N_COMPONENTS}_c=7_t=7"
    
    for category in ["tonic_neg", "phasic_pos"]: # Focus directly on Yun's two primary targets
        target_subfolder = os.path.join(SUMMARY_PLOTS_DIR, expt_ID, category)
        os.makedirs(target_subfolder, exist_ok=True)
        
        fig, ax = plt.subplots(figsize=(10, 12))
        ax.imshow(bg_canvas, cmap="gray", origin="upper", alpha=0.8)
        
        has_data = False
        
    
        for c_idx in range(1, 5):
            idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
            if not idx_file.exists(): continue
            
            cluster_indices = np.load(idx_file)
            valid_cluster_indices = cluster_indices[valid_mask[cluster_indices]]
            
            if len(valid_cluster_indices) == 0: continue
            has_data = True
            
            coords = medoids_vox[valid_cluster_indices]
            
          
            ax.scatter(
                coords[:, 0], 
                bg_canvas.shape[0] - coords[:, 1], 
                color=CONSOLIDATED_PALETTE[c_idx - 1],
                label=f"Circuit Network {c_idx} (n={len(valid_cluster_indices)})",
                s=3.0, 
                alpha=0.6,
                edgecolors='none'
            )
            
        if has_data:
            ax.set_title(f"Consolidated Functional Circuits Map — {LABEL_TITLES[category]}\n{expt_ID} (Aggregated K=4 Limits)", fontsize=13)
            ax.set_xlim(0, bg_canvas.shape[1])
            ax.set_ylim(bg_canvas.shape[0], 0)
            ax.axis("off")
            ax.legend(loc="upper right", markerscale=3.5, title="Functional Sub-Circuits")
            
            output_path = os.path.join(target_subfolder, "7_consolidated_anatomical_overlay.png")
            fig.savefig(output_path, dpi=250, bbox_inches='tight')
            print(f"  -> Consolidated overlay built successfully: {output_path}")
            
        plt.close(fig)
    gc.collect()

plt.ion()


### Verify Timeframe Shape 

In [ ]:

for fish_id in EXPT_FISH_LIST:
    fish = BASE_DIR / PROJ_ID / fish_id

    cluster_dir = BASE_DIR / PROJ_ID / fish_id / "FA_agglo_clustering_f={n_components}_c={t}_t={t}".format(n_components=N_COMPONENTS, t=N_CLUSTERS)
    category = "phasic_pos"
    model_path = cluster_dir / f"FA_model_{category}.joblib"

    try:
    
        fa_model = joblib.load(model_path)
    
        components_shape = fa_model.components_.shape
        n_factors_extracted = components_shape[0]
        n_timeframes_processed = components_shape[1]
    
        print(f"Verify time frames for {fish_id} ({category})")

        print(f"Successfully loaded trained model file from: {model_path.name}")
        print(f" -> matrix shape (raw): {components_shape}")
        print(f" -> number of hidden factors extracted: {n_factors_extracted}")
        print(f" -> number of timeframes: {n_timeframes_processed}")

        if n_timeframes_processed == 4500:
            print("The model analyzed the proper 4500 timeframes (baseline + drug).")
        else:
            print(f"The model processed a truncated timeline of {n_timeframes_processed} frames.")
        
    except FileNotFoundError:
        print(f"Error: Could not locate the model file at {model_path}. Ensure your script completed saving.")
    except Exception as e:
        print(f"Unexpected verification failure: {e}")


# Step 2 - Heatmap

In [ ]:
LABEL_TITLES = { 
    'tonic_pos': 'Tonic(+)', 
    'tonic_neg': 'Tonic(-)', 
    'phasic_pos': 'Phasic(+)', 
    'phasic_neg': 'Phasic(-)' 
}
CMAP = 'hot'
CLUSTER_COLORS = plt.cm.get_cmap('tab20', N_CLUSTERS)

def plot_4label_raster_fast(expt_ID):
    if expt_ID not in loaded_cluster_data:
        print(f"Skipping {expt_ID} — No data found in memory.")
        return None
        
    fig, axs = plt.subplots(2, 2, figsize=(16, 10))
    plt.suptitle(f'Clustered Neural Activity Heatmaps (Full Window)\n{expt_ID}', fontsize=16)
    
    fish_data = loaded_cluster_data[expt_ID]
    
    for idx, label in enumerate(RESPONSE_TYPES):
        ax = axs[idx // 2, idx % 2]
        cat_clusters = fish_data.get(label, {})
        
        trace_list = []
        cluster_id_list = []
    
        for c_idx in range(1, N_CLUSTERS + 1):
            if c_idx in cat_clusters:
                cluster_package = cat_clusters[c_idx]
                z_traces = cluster_package['full_timeline']
                
                trace_list.append(z_traces)
                cluster_id_list.extend([c_idx] * len(z_traces))
        
        if not trace_list:
            ax.text(0.5, 0.5, f"No responding cells found\nfor {LABEL_TITLES[label]}", 
                    ha='center', va='center', fontsize=12, color='gray')
            ax.set_title(LABEL_TITLES[label], fontsize=14)
            continue
            
        all_traces = np.vstack(trace_list)
        cluster_ids = np.array(cluster_id_list)
        
        sort_order = np.argsort(cluster_ids)
        sorted_traces = all_traces[sort_order][:, 0:]
        sorted_cluster_ids = cluster_ids[sort_order]
        
        im = ax.imshow(sorted_traces, aspect='auto', cmap=CMAP, interpolation='none', vmin=-3, vmax=6)
        
        ax.set_xlim(0, 7200)
        ax.set_title(f"{LABEL_TITLES[label]} (n = {sorted_traces.shape[0]} cells)", fontsize=14)
        ax.set_ylabel("Cells (Grouped by Cluster)", fontsize=12)
        ax.set_xlabel("All Volumes (Frames 0 - 7200)", fontsize=12)
        
        inset_ax = inset_axes(ax, width="2%", height="100%", loc='center left', 
                              bbox_to_anchor=(0.015, 0, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
        cluster_colors = [CLUSTER_COLORS(c - 1) for c in sorted_cluster_ids]
        color_strip = np.array(cluster_colors).reshape(len(cluster_colors), 1, 4)
        inset_ax.imshow(color_strip, aspect='auto')
        inset_ax.axis('off')
        
        cbar_ax = inset_axes(ax, width="1.5%", height="70%", loc='right', 
                             bbox_to_anchor=(0, 0.15, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
        plt.colorbar(im, cax=cbar_ax)
        
  
        for stim_time in [2700,5400]:
            ax.axvline(x=stim_time, color='white', linestyle='--', linewidth=1.0, alpha=0.8)
            
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    fig.subplots_adjust(wspace=0.2, hspace=0.25)
    return fig


# Save Graphs 

In [ ]:
#save heatmaps to a summary directory for easy access and reduces file size 
# where they are saved /ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary or /mnt/storage-raid10/Yun/analysis_output/chemogenetic/hcrt-trpv1_huc-h2b-g8m_csn_120min/unsupervised_plots_summary
SUMMARY_PLOTS_DIR = os.path.join("/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary")
os.makedirs(SUMMARY_PLOTS_DIR, exist_ok=True)

plt.ioff()
for expt_ID in EXPT_FISH_LIST:
    print(f"Generating heatmap for {expt_ID}...")  # Fixed space
    
    fig = plot_4label_raster_fast(expt_ID)

    if fig is not None:
        output_path = os.path.join(SUMMARY_PLOTS_DIR, f"{expt_ID}_pre_drug_heatmap_raster.png")
        
        if fig is not None:
            fig.savefig(output_path, dpi=200, bbox_inches='tight')
            
    if fig is not None:
        plt.close(fig)

plt.ion()



# Step 3 - Visualizing the spatial location of cells

In [ ]:
EXAMPLE_EXPT_FISH_LIST = ['251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4']

In [1]:
import os
import gc
from pathlib import Path
import numpy as np
import joblib
from scipy.stats import zscore
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import AgglomerativeClustering

BASE_DIR = Path("/mnt/storage-raid10/Yun/analysis_output/chemogenetic")
EXPT_PROJ = "hcrt-trpv1_huc-h2b-g8m_csn_120min"
CTRL_PROJ = "huc-h2b-g8m_csn_120min"

EXPT_FISH_LIST = [
    "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4", "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2", "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2", "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish3",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1", "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1"
]

CTRL_FISH_LIST = [
    "251008_huc-h2b-g8m_csn_10uM_fish1", "251008_huc-huc-h2b-g8m_csn_10uM_fish2",
    "251102_huc-h2b-g8m_csn_10uM_fish3", "251210_huc-h2b-g8m_csn_10uM_fish4",
    "251210_huc-h2b-g8m_csn_10uM_fish5", "260514_huc-h2b-g8m_csn_10uM_fish3",
    "260515_huc-h2b-g8m_csn_10uM_fish2"
]

N_COMPONENTS = 15
N_CLUSTERS = 7
PERCENTILE_CUTOFF = 1   # Set to 1 or 5
RESPONSE_TYPES = ["tonic_pos", "tonic_neg", "phasic_pos", "phasic_neg"]

def execute_dynamic_percentile_pipeline(proj_id, fish_list):
    print(f"\nRunning Pipeline for: {proj_id}")
    
    for expt_ID in fish_list:
        dir_expt = BASE_DIR / proj_id / expt_ID
        out_dir = dir_expt / f"FA_agglo_clustering_pct={PERCENTILE_CUTOFF}_c={N_CLUSTERS}"
        out_dir.mkdir(parents=True, exist_ok=True)
        
        for label in RESPONSE_TYPES:
            is_tonic = "tonic" in label
            data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
            data_path = dir_expt / data_filename
            
            if not data_path.exists(): continue
                
            try:
                traces = np.load(data_path)
                
    
                if is_tonic:
                    idx_file = dir_expt / f"{label}_glm_iaaft_nullp95_idxs.npy"
                    if not idx_file.exists(): continue
                    cell_idxs = np.load(idx_file)
                
                else:
                    dprime_path = dir_expt / "phasic_dprime_cells_raw.npy"
                    if not dprime_path.exists(): continue
                    dprime = np.load(dprime_path)
                    
                    abs_dprime = np.abs(dprime)
                    threshold_value = np.percentile(abs_dprime, 100 - PERCENTILE_CUTOFF)
                    
                    if "pos" in label:
                        cell_idxs = np.where(dprime >= threshold_value)[0]
                    else:
                        cell_idxs = np.where(dprime <= -threshold_value)[0]
                
                if len(cell_idxs) < N_CLUSTERS: continue
   
                trace_subset = traces[cell_idxs, 2700:7200]
                z_traces = zscore(trace_subset, axis=1)
                
                valid_mask = np.all(np.isfinite(z_traces), axis=1)
                z_traces = z_traces[valid_mask]
                cell_idxs = cell_idxs[valid_mask]

                fa = FactorAnalysis(n_components=N_COMPONENTS, random_state=0)
                latent = fa.fit_transform(z_traces)
                joblib.dump(fa, out_dir / f"FA_model_{label}.joblib")
                
                agglo = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward')
                cluster_labels = agglo.fit_predict(latent) + 1
                
                for i in range(N_CLUSTERS):
                    cluster_indices = cell_idxs[cluster_labels == (i + 1)]
                    np.save(out_dir / f"{label}_c{i+1}_idxs.npy", cluster_indices)
                    
                print(f"    {expt_ID} — {label}: Logged top {PERCENTILE_CUTOFF}% allocation pool cleanly.")
                
                del traces, trace_subset, z_traces, latent
                gc.collect()
                
            except Exception as e:
                print(f"Failed compiling processing pass for {expt_ID} -> {label}: {e}")
                continue

if __name__ == "__main__":
    execute_dynamic_percentile_pipeline(EXPT_PROJ, EXPT_FISH_LIST)
    execute_dynamic_percentile_pipeline(CTRL_PROJ, CTRL_FISH_LIST)
    print("\nProcessing complete! All percentage-normalized clusters are written to disk.")



Running Pipeline for: hcrt-trpv1_huc-h2b-g8m_csn_120min


OSError: [Errno 112] Host is down: '/mnt/storage-raid10/Yun/analysis_output/chemogenetic/hcrt-trpv1_huc-h2b-g8m_csn_120min/251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4/FA_agglo_clustering_pct=1_c=7'

In [ ]:
TEMPLATE_BRAIN_PATH = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz"
SUMMARY_PLOTS_DIR = "/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary"
COLOR_PALETTE = plt.cm.get_cmap('tab10', N_CLUSTERS)
if not os.path.exists(TEMPLATE_BRAIN_PATH):
    raise FileNotFoundError(f"Missing mandatory anatomical canvas template: {TEMPLATE_BRAIN_PATH}")

nii_obj = nib.load(TEMPLATE_BRAIN_PATH)
template_img = nii_obj.get_fdata()

bg_canvas = np.max(template_img, axis=2).T
bg_canvas = np.flipud(bg_canvas)
plt.ioff()

for expt_ID in EXPT_FISH_LIST:
    if expt_ID not in loaded_cluster_data:
        continue
        
    print(f"\nProjecting anatomical spatial profiles for: {expt_ID}")
    fish_dir = BASE_DIR / PROJ_ID / expt_ID
    
    vox_path = fish_dir / "medoids_template_vox.npy"
    if not vox_path.exists():
        print(f"  Missing template voxels for {expt_ID}. Skipping.")
        continue
        
    medoids_vox = np.load(vox_path)
    valid_registration_mask = (medoids_vox >= 0).all(axis=1)

    cluster_dir = fish_dir / f"FA_agglo_clustering_f={N_COMPONENTS}_c={N_CLUSTERS}_t={N_CLUSTERS}"
    
    for category in RESPONSE_TYPES:
        target_subfolder = os.path.join(SUMMARY_PLOTS_DIR, expt_ID, category)
        os.makedirs(target_subfolder, exist_ok=True)
        

        valid_cluster_files = []
        for c_idx in range(1, N_CLUSTERS + 1):
            idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
            if idx_file.exists():
                cluster_indices = np.load(idx_file)
                if len(cluster_indices) > 0:
                    valid_cluster_indices = cluster_indices[valid_registration_mask[cluster_indices]]
                    if len(valid_cluster_indices) > 0:
                        valid_cluster_files.append((c_idx, idx_file, valid_cluster_indices))
                        
        n_valid_panels = len(valid_cluster_files)
        if n_valid_panels == 0:
            continue
     
        n_cols = min(6, n_valid_panels)
        n_rows = int(np.ceil(n_valid_panels / n_cols))
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows), squeeze=False)
        plt.suptitle(f"Spatial Distribution Plot — {RESPONSE_TYPES[category]}\n{expt_ID} ( K={N_CLUSTERS})", fontsize=15, y=0.98)
        axes = axes.flatten()
        
        plotted_panels = 0
        

        for c_idx, idx_file, valid_cluster_indices in valid_cluster_files:
  
            if plotted_panels >= len(axes):
                print(f"Omitted extra cluster panel {c_idx} to avoid array out-of-bounds")
                break
                
 
            coords = medoids_vox[valid_cluster_indices]
            
            ax = axes[plotted_panels]
  
            ax.imshow(bg_canvas, cmap="gray", origin="lower")
 
            ax.scatter(
                coords[:, 0], 
                coords[:, 1], 
                color=COLOR_PALETTE(c_idx - 1),
                s=2.5, 
                alpha=0.6, 
                edgecolors='none'
            )
            
            ax.set_title(f"Cluster {c_idx} (n={len(valid_cluster_indices)} cells)", fontsize=11)
            ax.set_xlim(0, bg_canvas.shape[1])
            ax.set_ylim(0, bg_canvas.shape[0]) 
            ax.axis("off")
            ax.text(0.05, 0.08, f"C{c_idx}", 
                    transform=ax.transAxes, 
                    color="white", 
                    fontsize=20, 
                    weight="bold", 
                    ha="left", 
                    va="bottom")
            plotted_panels += 1
            

        for empty_idx in range(plotted_panels, len(axes)):
            fig.delaxes(axes[empty_idx])
            
        if plotted_panels > 0:
            fig.subplots_adjust(wspace=0.1, hspace=0.1)
            output_path = os.path.join(target_subfolder, "spatial_distrbution_plot.png")
            fig.savefig(output_path, dpi=200, bbox_inches='tight', pad_inches=0.5)
            print(f"Saved plot for: {output_path}")
            
        plt.close(fig)
        
    gc.collect()

plt.ion()
print("\nAll plots rendered")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm
from tqdm import tqdm

def plot_cluster_xy_projection(
    expt_ID_list,
    proj_ID,
    dir_ants_output,
    medoid_types,
    stim_start,
    stim_end,
    use_qc,
    top_n_clusters,
    brain_shape=(597, 974, 359),
    save_fig=True
):


    for expt_ID in tqdm(expt_ID_list, desc="Processing experiments"):
        suffix = "_qc.npy" if use_qc else ".npy"
        fig_base_dir = os.path.join(dir_ants_output, proj_ID, expt_ID, f"FA_agglo_clustering_f={N_COMPONENTS}_c={N_CLUSTERS}")
        os.makedirs(fig_base_dir, exist_ok=True)

    
        print(f"\n🧠 Processing: {expt_ID}")
        dir_expt = os.path.join(dir_ants_output, proj_ID, expt_ID)
        coord_path = os.path.join(dir_expt, "transformed_all_cells.npy")
        volume_path = os.path.join(dir_expt, "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.g")
        
        volume_img = ants.image_read(volume_path)
        volume_img = volume_img.numpy() 
        # Compute max-projection, then fix orientation
        bg = np.max(volume_img, axis=2).T
        bg = np.flipud(bg)  # Flip vertically to match dorsal-up orientation
        
        


        if not os.path.exists(coord_path) or not os.path.exists(volume_path):
            print("❌ Missing coordinate or registered volume.")
            continue

        all_coords = np.load(coord_path)
        for label in medoid_types:
            print(f"➤ Label: {label}")
            is_neg = "neg" in label
            is_tonic = "tonic" in label
            data_file = os.path.join(dir_expt, f"data_array_f_{'tonic' if is_tonic else 'phasic'}.npy")
            if not os.path.exists(data_file):
                print("❌ Missing data file.")
                continue
            data_array = np.load(data_file)
            z_traces = (data_array - np.nanmean(data_array, axis=1, keepdims=True)) / np.nanstd(data_array, axis=1, keepdims=True)

            fig, axes = plt.subplots(3, 5, figsize=(20, 15))
            axes = axes.flatten()
            plotted = 0
            y_max = brain_shape[1]

            for i in range(1, top_n_clusters + 1):
                clust_path = os.path.join(dir_expt, "FA_agglo_clustering", f"{label}_c{i}_idxs.npy")
                if not os.path.exists(clust_path):
                    print(f"⚠️ Missing: {clust_path}")
                    continue
                try:
                    idxs = np.load(clust_path, allow_pickle=True)
                    idxs = np.array(idxs).flatten()
                except Exception as e:
                    print(f"❌ Error loading {clust_path}: {e}")
                    continue

                valid_mask = (idxs < len(z_traces)) & (idxs < len(all_coords))
                idxs = idxs[valid_mask]
                if len(idxs) == 0:
                    continue

                coords = all_coords[idxs]
                vals = np.nanmean(z_traces[idxs, stim_start:stim_end], axis=1)
                if is_neg:
                    vals = -vals
                norm = Normalize(vmin=0, vmax=0.8 if "tonic" in label else 0.4)
                cmap = cm.get_cmap("Reds" if "pos" in label else "Blues")

                # Plot
                ax = axes[plotted]
                ax.set_title(f"{label} — Cluster {i} (n={len(idxs)})")
                #ax.imshow(np.max(volume_img, axis=2).T, cmap="gray", origin="upper")
                
                ax.imshow(bg, cmap="gray", origin="upper")

                sc = ax.scatter(coords[:, 0], y_max - coords[:, 1], c=vals, cmap=cmap, norm=norm, s=3, alpha=0.7)
                plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02, label="Z-score")
                ax.set_xlim(0, brain_shape[0])
                ax.set_ylim(0, brain_shape[1])
                ax.set_xticks([])
                ax.set_yticks([])
                plotted += 1
                if plotted >= 15:
                    break

            # hide unused subplots
            for j in range(plotted, 8):
                axes[j].axis("off")

            plt.tight_layout()
            if save_fig:
                save_path = os.path.join(fig_base_dir, f"{label}_cluster_distribution.pdf")
                plt.savefig(save_path, dpi=300)
                print(f"📄 Saved: {save_path}")
            plt.show()


plot_cluster_xy_projection(
    EXPT_FISH_LIST,
    PROJ_ID,
    DIR_ANTS_OUTPUT,
    RESPONSE_TYPES,
    stim_start=1800,
    stim_end=3600,
    use_qc=False,  # or False
    top_n_clusters=15
)


In [ ]:
import os
import gc
from pathlib import Path
import numpy as np
from scipy.stats import zscore
from concurrent.futures import ThreadPoolExecutor

BASE_DIR = Path("/mnt/storage-raid10/Yun/analysis_output/chemogenetic")
PROJ_ID = "hcrt-trpv1_huc-h2b-g8m_csn_120min"

EXPT_FISH_LIST = [
    "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2", 
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish3",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1"
]

N_COMPONENTS = 15
N_CLUSTERS = 7
RESPONSE_TYPES = ["tonic_pos", "tonic_neg", "phasic_pos", "phasic_neg"]
stim_start_post = 2700
stim_end_post = 7200

loaded_cluster_data = {fish: {cat: {} for cat in RESPONSE_TYPES} for fish in EXPT_FISH_LIST}

def process_fish_analysis(fish_id):
    fish_dir = BASE_DIR / PROJ_ID / fish_id
    cluster_dir = fish_dir / f"FA_agglo_clustering_f={N_COMPONENTS}_c={N_CLUSTERS}_t={N_CLUSTERS}"
    
    if not cluster_dir.exists():
        print(f"Skipping {fish_id} — Path not found: {cluster_dir.name}")
        return None
        
    fish_results = {cat: {} for cat in RESPONSE_TYPES}
    
    for category in RESPONSE_TYPES:
        is_tonic = "tonic" in category
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = fish_dir / data_filename
        
        if not data_path.exists():
            print(f'data path does not exist for {data_path}')
            continue
            
        try:
            all_cluster_cells = []
            cluster_cell_mappings = {}
            
            for c_idx in range(1, N_CLUSTERS + 1):
                idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
                if idx_file.exists():
                    c_cells = np.load(idx_file)
                    if len(c_cells) > 0:
                        all_cluster_cells.append(c_cells)
                        cluster_cell_mappings[c_idx] = c_cells
                else: print(f'idx file for {idx_file} does not exist')
                        
            if not all_cluster_cells:
                continue
                
            total_responders = np.concatenate(all_cluster_cells)
            unique_responders, inverse_mapping = np.unique(total_responders, return_inverse=True)
    
            raw_traces_mmap = np.load(data_path, mmap_mode="r")
            
            pooled_post_traces = np.array(raw_traces_mmap[unique_responders, stim_start_post:stim_end_post])
            pooled_full_traces = np.array(raw_traces_mmap[unique_responders, 0:7200])
            
            current_idx_pointer = 0
            for c_idx in range(1, N_CLUSTERS + 1):
                if c_idx not in cluster_cell_mappings:
                    continue
                    
                n_cells_in_cluster = len(cluster_cell_mappings[c_idx])
                cluster_inverse_subset = inverse_mapping[current_idx_pointer : current_idx_pointer + n_cells_in_cluster]
                current_idx_pointer += n_cells_in_cluster
                
                c_post = pooled_post_traces[cluster_inverse_subset]
                c_full = pooled_full_traces[cluster_inverse_subset]
                z_post = zscore(c_post, axis=1)
                z_full = zscore(c_full, axis=1)
                
                valid_mask = np.all(np.isfinite(z_post), axis=1) & np.all(np.isfinite(z_full), axis=1)
                
                if np.sum(valid_mask) > 0:
                    fish_results[category][c_idx] = {
                        'post_drug': z_post[valid_mask],
                        'full_timeline': z_full[valid_mask]
                    }
                    
        except Exception as e:
            print(f"Error loading {category} for {fish_id}: {e}")
            continue
            
    return fish_id, fish_results

print("Loading Dual-Window post-drug and full-timeline matrices from disk...")
with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish_analysis, EXPT_FISH_LIST)

for result in results:
    if result is not None:
        fish_id, fish_results = result
        loaded_cluster_data[fish_id] = fish_results
        
print("Data loading completed successfully!")


In [ ]:
import gc
from concurrent.futures import ThreadPoolExecutor
import os
from pathlib import Path
import numpy as np
from scipy.stats import zscore

BASE_DIR = Path("/mnt/storage-raid10/Yun/analysis_output/chemogenetic")
PROJ_ID = "hcrt-trpv1_huc-h2b-g8m_csn_120min"

EXPT_FISH_LIST = [
    "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish3",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
]

N_COMPONENTS = 15
N_CLUSTERS = 7
RESPONSE_TYPES = ["tonic_pos", "tonic_neg", "phasic_pos", "phasic_neg"]
stim_start_post = 2700
stim_end_post = 7200
RELATIVE_PERCENTILE = 0.95  # Top 5% threshold

loaded_cluster_data = {
    fish: {cat: {} for cat in RESPONSE_TYPES} for fish in EXPT_FISH_LIST
}


def process_fish_analysis(fish_id):
    fish_dir = BASE_DIR / PROJ_ID / fish_id
    cluster_dir = (
        fish_dir
        / f"FA_agglo_clustering_f={N_COMPONENTS}_c={N_CLUSTERS}_t={N_CLUSTERS}"
    )

    if not cluster_dir.exists():
        print(f"Skipping {fish_id} — Path not found: {cluster_dir.name}")
        return None

    fish_results = {cat: {} for cat in RESPONSE_TYPES}

    for category in RESPONSE_TYPES:
        is_tonic = "tonic" in category
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = fish_dir / data_filename

        if not data_path.exists():
            print(f"data path does not exist for {data_path}")
            continue

        try:
            all_cluster_cells = []
            cluster_cell_mappings = {}

            for c_idx in range(1, N_CLUSTERS + 1):
                idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
                if idx_file.exists():
                    c_cells = np.load(idx_file)
                    if len(c_cells) > 0:
                        all_cluster_cells.append(c_cells)
                        cluster_cell_mappings[c_idx] = c_cells
                else:
                    print(f"idx file for {idx_file} does not exist")

            if not all_cluster_cells:
                continue

            total_responders = np.concatenate(all_cluster_cells)
            unique_responders, inverse_mapping = np.unique(
                total_responders, return_inverse=True
            )

            raw_traces_mmap = np.load(data_path, mmap_mode="r")

            pooled_post_traces = np.array(
                raw_traces_mmap[unique_responders, stim_start_post:stim_end_post]
            )
            pooled_full_traces = np.array(
                raw_traces_mmap[unique_responders, 0:7200]
            )

            # =================================================================
            # NEW: CALCULATE RELATIVE PERCENTILE THRESHOLD FOR THIS FISH
            # =================================================================
            # 1. Use the mean response intensity of each cell during the drug window
            #    (Use np.abs if you want to capture magnitude for both pos and neg clusters)
            if "pos" in category:
                cell_intensities = np.mean(pooled_post_traces, axis=1)
            else:  # For negative responders, look at the strongest drops
                cell_intensities = np.abs(np.mean(pooled_post_traces, axis=1))

            # 2. Convert to Z-scores across all responders in this fish to standardize metrics
            fish_responder_z = zscore(cell_intensities)

            # 3. Determine the 95th percentile cutoff value for this specific fish
            #    Handle cases where all values might be NaN safely
            if np.any(np.isfinite(fish_responder_z)):
                cutoff_threshold = np.nanpercentile(
                    fish_responder_z, RELATIVE_PERCENTILE * 100
                )
                # 4. Create a boolean mask of cells that pass the top 5% criteria
                relative_top_5_mask = fish_responder_z >= cutoff_threshold
            else:
                relative_top_5_mask = np.zeros_like(
                    unique_responders, dtype=bool
                )
            # =================================================================

            current_idx_pointer = 0
            for c_idx in range(1, N_CLUSTERS + 1):
                if c_idx not in cluster_cell_mappings:
                    continue

                n_cells_in_cluster = len(cluster_cell_mappings[c_idx])
                cluster_inverse_subset = inverse_mapping[
                    current_idx_pointer : current_idx_pointer
                    + n_cells_in_cluster
                ]
                current_idx_pointer += n_cells_in_cluster

                # Get this cluster's traces
                c_post = pooled_post_traces[cluster_inverse_subset]
                c_full = pooled_full_traces[cluster_inverse_subset]

                # Extract the relative top 5% mask specific to this cluster's indices
                c_relative_mask = relative_top_5_mask[cluster_inverse_subset]

                z_post = zscore(c_post, axis=1)
                z_full = zscore(c_full, axis=1)

                # Combine your original finite check with our new relative threshold mask
                valid_mask = (
                    np.all(np.isfinite(z_post), axis=1)
                    & np.all(np.isfinite(z_full), axis=1)
                    & c_relative_mask
                )

                if np.sum(valid_mask) > 0:
                    fish_results[category][c_idx] = {
                        "post_drug": z_post[valid_mask],
                        "full_timeline": z_full[valid_mask],
                    }

        except Exception as e:
            print(f"Error loading {category} for {fish_id}: {e}")
            continue

    return fish_id, fish_results


print("Loading Dual-Window post-drug and full-timeline matrices from disk...")
with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish_analysis, EXPT_FISH_LIST)

for result in results:
    if result is not None:
        fish_id, fish_results = result
        loaded_cluster_data[fish_id] = fish_results

print("Data loading completed successfully!")


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import nibabel as nib

TEMPLATE_BRAIN_PATH = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz"
SUMMARY_PLOTS_DIR = "/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary"

CATEGORY_COLORS = {
    'tonic_pos': 'orange',
    'tonic_neg': 'blue',
    'phasic_pos': 'red',    
    'phasic_neg': 'purple'
}

LABEL_TITLES = { 
    'tonic_pos': 'Tonic(+)', 
    'tonic_neg': 'Tonic(-)', 
    'phasic_pos': 'Phasic(+)', 
    'phasic_neg': 'Phasic(-)' 
}

print("Initializing Global Reference Brain Canvas for Raw Responders...")
if not os.path.exists(TEMPLATE_BRAIN_PATH):
    raise FileNotFoundError(f"Missing mandatory anatomical canvas template: {TEMPLATE_BRAIN_PATH}")

nii_obj = nib.load(TEMPLATE_BRAIN_PATH)
template_img = nii_obj.get_fdata()
bg_canvas = np.max(template_img, axis=2).T
bg_canvas = np.flipud(bg_canvas)

plt.ioff()

print("Generating Pre-Clustering Raw Responder Map Overlays...")

for expt_ID in EXPT_FISH_LIST:
    print(f"\nEvaluating raw responder maps for: {expt_ID}")
    fish_dir = BASE_DIR / PROJ_ID / expt_ID
    
    vox_path = fish_dir / "medoids_template_vox.npy"
    if not vox_path.exists():
        print(f"  Warning: {vox_path} missing. Skipping this fish.")
        continue
    medoids_vox = np.load(vox_path)
    
    # Establish a safe in-bounds registration mask
    valid_registration_mask = (medoids_vox >= 0).all(axis=1)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    plt.suptitle(f"Pre-Clustering Whole Responder Distributions\n{expt_ID} (Tonic: p95 | Phasic: ±0.25 Cutoff)", fontsize=15, y=0.98)
    axes = axes.flatten()
    
    for idx, category in enumerate(RESPONSE_TYPES):
        ax = axes[idx]
        
        is_tonic = "tonic" in category
        if is_tonic:
            idx_path = fish_dir / f"{category}_glm_iaaft_nullp95_idxs.npy"
            if not idx_path.exists():
                ax.text(0.5, 0.5, f"Missing index file:\n{category}_nullp95", ha='center', va='center', color='gray')
                ax.axis("off")
                continue
            raw_responder_idxs = np.load(idx_path)
        else:
            dprime_path = fish_dir / "phasic_dprime_cells_raw.npy"
            if not dprime_path.exists():
                ax.text(0.5, 0.5, f"Missing dprime file:\nphasic_dprime_cells_raw.npy", ha='center', va='center', color='gray')
                ax.axis("off")
                continue
            dprime = np.load(dprime_path)
            if "pos" in category:
                raw_responder_idxs = np.where(dprime >= 0.25)[0]
            else:
                raw_responder_idxs = np.where(dprime <= -0.25)[0]
        
        valid_raw_idxs = raw_responder_idxs[valid_registration_mask[raw_responder_idxs]]
        total_cells_found = len(valid_raw_idxs)
        
        ax.imshow(bg_canvas, cmap="gray", origin="lower")
        
        if total_cells_found == 0:
            ax.text(0.5, 0.5, f"No Responding Cells\n(n = 0)", ha='center', va='center', color='gray')
            ax.set_title(LABEL_TITLES[category], fontsize=12)
            ax.axis("off")
            continue
            
        coords = medoids_vox[valid_raw_idxs]
        
        ax.scatter(
            coords[:, 0], 
            coords[:, 1], 
            color=CATEGORY_COLORS[category],
            s=2.5,
            alpha=0.6, 
            edgecolors='none'
        )
        
        ax.set_title(f"{LABEL_TITLES[category]} Responder Pool (n = {total_cells_found} cells)", fontsize=13)
        ax.set_xlim(0, bg_canvas.shape[1])
        ax.set_ylim(bg_canvas.shape[0], 0)
        ax.axis("off")

    fig.subplots_adjust(wspace=0.3, hspace=0.3)
    fish_summary_dir = os.path.join(SUMMARY_PLOTS_DIR, expt_ID)
    os.makedirs(fish_summary_dir, exist_ok=True)
    
    output_path = os.path.join(fish_summary_dir, "whole_pool_responder_distribution.png")
    fig.savefig(output_path, dpi=200, bbox_inches='tight', pad_inches=0.4)
    print(f"  Successfully saved unified raw pool grid -> {output_path}")
    
    plt.close(fig)
    gc.collect()

plt.ion()
print("\nAll intermediate whole pool responder distribution maps rendered and linked inside your sidebar workspace!")


In [ ]:
tonic_pos[]